In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

import healpy as hp

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
min_nobs = 2

In [4]:
randoms_density = 2500

randoms_columns = ['RA', 'DEC', 'NOBS_G', 'NOBS_R', 'NOBS_Z', 'MASKBITS', 'PHOTSYS', 'EBV']
randoms = Table(fitsio.read('/Users/rongpu/Documents/Data/dr9/randoms/randoms-1-0.fits', columns=randoms_columns))
lrgmask = Table(fitsio.read('/Users/rongpu/Documents/Data/dr9/randoms/randoms-1-0-lrgmask_v1.fits'))
randoms = hstack([randoms, lrgmask], join_type='exact')

mask = (randoms['NOBS_G']>=min_nobs) & (randoms['NOBS_R']>=min_nobs) & (randoms['NOBS_Z']>=min_nobs)
print('NOBS', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
randoms = randoms[mask]

mask = randoms['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
randoms = randoms[mask]

# Martin's EBV cut
mask = randoms['EBV']<0.15
print('EBV', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
randoms = randoms[mask]

# Martin's STARDENS cut
stardens = np.load('/Users/rongpu/Documents/Data/desi_lrg_selection/dr7/healpix_maps/pixweight-dr7.1-0.22.0_stardens_64_ring.npy')
stardens_nside = 64
mask = stardens>=2500
bad_hp_idx = np.arange(len(stardens))[mask]
cat_hp_idx = hp.pixelfunc.ang2pix(stardens_nside, randoms['RA'], randoms['DEC'], lonlat=True, nest=False)
mask_bad = np.in1d(cat_hp_idx, bad_hp_idx)
print('STARDENS', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
randoms = randoms[~mask_bad]

print(len(randoms))

area = len(randoms)/randoms_density
area_north = np.sum(randoms['PHOTSYS']=='N')/randoms_density
area_south = np.sum(randoms['PHOTSYS']=='S')/randoms_density
print('Area:', area)
print('Area (north):', area_north)
print('Area (south):', area_south)

NOBS 47030705 4707911 0.9090058574431137
LRG mask 43072333 3958372 0.9158343044187834
EBV 42566848 505485 0.01173572371851787
STARDENS 41640782 926066 0.021755569028742743
41640782
Area: 16656.3128
Area (north): 4212.9212
Area (south): 12443.3916


In [5]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/lrg_xcorr/catalogs/main_lrg_minobs_1_20210913.fits', columns=['TARGETID', 'NOBS_G', 'NOBS_R', 'NOBS_Z', 'Z_PHOT_MEDIAN', 'pz_bin', 'lrg_mask', 'EBV', 'RA', 'DEC', 'PHOTSYS']))
print(len(cat))

mask = (cat['NOBS_G']>=min_nobs) & (cat['NOBS_R']>=min_nobs) & (cat['NOBS_Z']>=min_nobs)
print('NOBS', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

mask = cat['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]

# Martin's EBV cut
mask = cat['EBV']<0.15
print('EBV', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Martin's STARDENS cut
stardens = np.load('/Users/rongpu/Documents/Data/desi_lrg_selection/dr7/healpix_maps/pixweight-dr7.1-0.22.0_stardens_64_ring.npy')
stardens_nside = 64
mask = stardens>=2500
bad_hp_idx = np.arange(len(stardens))[mask]
cat_hp_idx = hp.pixelfunc.ang2pix(stardens_nside, cat['RA'], cat['DEC'], lonlat=True, nest=False)
mask_bad = np.in1d(cat_hp_idx, bad_hp_idx)
print('STARDENS', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
cat = cat[~mask_bad]

print(len(cat))

# Add DESI_TARGET column
main_dir = '/Users/rongpu/Documents/Data/desi_targets/dr9.0/combined/1.0.0'
lrg = []
for field in ['north', 'south']:
    lrg.append(Table(fitsio.read(os.path.join(main_dir, 'dr9_lrg_{}_1.0.0_basic.fits'.format(field)), columns=['TARGETID', 'DESI_TARGET'])))
lrg = vstack(lrg)
cat = join(cat, lrg, keys='TARGETID')
print(len(cat))

12338990
NOBS 11786823 552167 0.9552502271255588
LRG mask 10364959 1421864 0.8793683420884492
EBV 10233894 131065 0.012645009015472226
STARDENS 10002070 231824 0.02265256997971642
10002070
10002070


In [6]:
mask_north = cat['PHOTSYS']=='N'
mask_south = cat['PHOTSYS']=='S'

print('Density: {:.2f}'.format(len(cat)/area))
print('Density (north): {:.2f}'.format(np.sum(mask_north)/area_north))
print('Density (south): {:.2f}'.format(np.sum(mask_south)/area_south))

Density: 600.50
Density (north): 606.51
Density (south): 598.46


In [7]:
# QSO overlap
mask = cat['DESI_TARGET'] & 2**2>0
print('QSO overlap: {:.3f}%'.format(np.sum(mask)/len(cat)*100))
print('QSO overlap (north): {:.3f}%'.format(np.sum(mask & mask_north)/np.sum(mask_north)*100))
print('QSO overlap (south): {:.3f}%'.format(np.sum(mask & mask_south)/np.sum(mask_south)*100))

QSO overlap: 0.611%
QSO overlap (north): 0.554%
QSO overlap (south): 0.631%


In [8]:
densities = {}
for pz_bin in range(1, 5):
    mask = mask_south & (cat['pz_bin']==pz_bin)
    densities['bin_{}_south'.format(pz_bin)] = np.sum(mask)/area_south
    mask = mask_north & (cat['pz_bin']==pz_bin)
    densities['bin_{}_north'.format(pz_bin)] = np.sum(mask)/area_north

In [9]:
densities

{'bin_1_south': 82.7460899004416,
 'bin_1_north': 84.14565171548901,
 'bin_2_south': 148.14232801288676,
 'bin_2_north': 150.72036002002602,
 'bin_3_south': 160.62839330717517,
 'bin_3_north': 164.0460305784974,
 'bin_4_south': 146.88800760718644,
 'bin_4_north': 151.32279236554436}